# HIMNETRA - Part 2: Flood propagation (LAHARZ-style routing on a 30 m DEM)

Run **after Part 1** (needs `himnetra/out/lakes.json` on Drive). Run cells top to bottom; **do cell 6 (one lake) and look at the picture before cell 7**.

Method: lake volume from area (Huggel 2002) -> scenario release (10/30/60/100 %) -> cross-section area A = 0.05 V^(2/3) and planimetric area B = 200 V^(2/3) (Iverson 1998 LAHARZ coefficients) -> flood follows the D8 flow path from the lake outlet on the Copernicus 30 m DEM -> travel time from Manning's equation (n = 0.06) -> settlements and roads from OpenStreetMap.

This is a simplified, scenario-level model, **not** a hydrodynamic simulation. Say so in the report.

Outputs (Drive `himnetra/out/`): `flood_results.json`, `flood_polygons.geojson`, `flood_backtest.json`.

In [ ]:
#@title 1) Install + settings
!pip -q install pysheds shapely pyproj rasterio requests

PROJECT = "YOUR_PROJECT_ID"  #@param {type:"string"}
BASE = "/content/drive/MyDrive/himnetra"      # same Drive folder as Part 1 (needs out/lakes.json)
DEM_HALF_PX = 1000    # DEM window = 2000 x 2000 px of 30 m (30 km radius around each lake)
MAX_RUN_KM = 30       # length of the downstream reach that is modelled
ONLY = []             # e.g. ["GL-HP-001", "GL-HP-002"] to run a few lakes; [] = all lakes in lakes.json
FORCE = False         # False = resume: skips lakes already finished with the new OSM logic; True = recompute everything
print("settings ok")


In [ ]:
#@title 2) Drive + Earth Engine + load lakes
import os, io, json, glob, math, time, datetime, warnings
import numpy as np
warnings.filterwarnings("ignore")
from google.colab import drive
drive.mount("/content/drive")
import ee
assert PROJECT and PROJECT != "YOUR_PROJECT_ID", "❌ Put your real Google Cloud Project ID in cell 1 and re-run it"
try: ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
OUT = f"{BASE}/out"
lakes = json.load(open(f"{OUT}/lakes.json"))
print("✅ Earth Engine OK;", len(lakes), "lakes loaded from", OUT)
assert len(lakes) >= 5, "❌ lakes.json looks wrong - run Part 1 first"


In [ ]:
#@title 3) Flood model functions (just run)
import json, math, time
import numpy as np
from scipy import ndimage as ndi
from scipy.ndimage import map_coordinates
from scipy.spatial import cKDTree

FRACTION = {"MINOR": 0.10, "MODERATE": 0.30, "MAJOR": 0.60, "EXTREME": 1.00}   # share of the lake released
A_COEF, B_COEF = 0.05, 200.0      # LAHARZ (Iverson et al. 1998, lahar coefficients): cross-section A=0.05 V^(2/3), planimetric B=200 V^(2/3)
MANNING_N = 0.06                  # steep boulder channels
STEP = {64: (-1, 0), 128: (-1, 1), 1: (0, 1), 2: (1, 1), 4: (1, 0), 8: (1, -1), 16: (0, -1), 32: (-1, -1)}

def lake_volume_m3(area_km2):     # Huggel et al. 2002: V[m3] = 0.104 * A[m2]^1.42
    return 0.104 * (max(area_km2, 0.0) * 1e6) ** 1.42

def peak_discharge(v_m3):         # Evans 1986: Qp = 0.72 * V^0.53
    return 0.72 * v_m3 ** 0.53

def condition_dem(dem, affine, epsg):
    """Fill depressions + flow directions (D8) with pysheds. Returns filled DEM and fdir as numpy."""
    import pyproj
    from pysheds.grid import Grid
    from pysheds.sview import Raster, ViewFinder
    vf = ViewFinder(affine=affine, shape=dem.shape, crs=pyproj.CRS.from_epsg(epsg), nodata=np.float32(-9999))
    r = Raster(dem.astype("float32"), vf); grid = Grid(viewfinder=vf)
    inf = grid.resolve_flats(grid.fill_depressions(grid.fill_pits(r)))
    return np.asarray(inf, dtype="float32"), np.asarray(grid.flowdir(inf))

def trace_path(fdir, r, c, max_px):
    H, W = fdir.shape; path, seen = [(r, c)], {(r, c)}
    while len(path) < max_px:
        k = int(fdir[r, c])
        if k not in STEP: break
        dr, dc = STEP[k]; r, c = r + dr, c + dc
        if not (0 <= r < H and 0 <= c < W) or (r, c) in seen: break
        path.append((r, c)); seen.add((r, c))
    return path

def inundate(dem, path, px, v_m3, half_w=70, n=MANNING_N, skip_m=0.0):
    """LAHARZ-style: for every cross-section find the stage whose flow area equals A, until the planimetric area B is used up."""
    H, W = dem.shape; Ac = A_COEF * v_m3 ** (2 / 3); Bmax = B_COEF * v_m3 ** (2 / 3)
    P = np.array(path, float); N = len(P); mask = np.zeros(dem.shape, bool)
    ts = np.arange(-half_w, half_w + 1.0); c0 = half_w
    bed = np.zeros(N); width = np.zeros(N); depth = np.zeros(N); ds = np.zeros(N); cum = 0.0; used = N
    ds[1:] = px * np.hypot(*(P[1:] - P[:-1]).T); cumd = np.cumsum(ds)
    for i in range(N):
        j0, j1 = max(0, i - 4), min(N - 1, i + 4); d = P[j1] - P[j0]; L = math.hypot(*d)
        if cumd[i] < skip_m:                      # still inside the lake basin: no flood polygon here
            bed[i] = dem[path[i]]; continue
        if L == 0: continue
        d /= L; perp = np.array([-d[1], d[0]])
        rr, cc = P[i, 0] + ts * perp[0], P[i, 1] + ts * perp[1]
        ok = (rr >= 0) & (rr <= H - 1) & (cc >= 0) & (cc <= W - 1)
        z = map_coordinates(dem, [rr, cc], order=1, mode="nearest"); z[~ok] = np.inf
        b = z[c0 - 1:c0 + 2].min(); bed[i] = b
        def flow_area(h):
            lvl = b + h; wet = z <= lvl
            keep = np.zeros_like(wet); l = np.cumprod(wet[c0::-1]).astype(bool); r_ = np.cumprod(wet[c0:]).astype(bool)
            keep[c0::-1] |= l; keep[c0:] |= r_
            return float(((lvl - z) * keep)[np.isfinite(z)].sum() * px), keep
        lo, hi = 0.0, 200.0
        if flow_area(hi)[0] < Ac: h = hi
        else:
            for _ in range(30):
                mid = (lo + hi) / 2
                if flow_area(mid)[0] < Ac: lo = mid
                else: hi = mid
            h = hi
        _, keep = flow_area(h); keep &= ok
        mask[np.round(rr[keep]).astype(int), np.round(cc[keep]).astype(int)] = True
        width[i] = keep.sum() * px; depth[i] = h; cum += width[i] * ds[i]
        if cum > Bmax: used = i + 1; break
    mask = ndi.binary_closing(mask, iterations=1)
    k = 6; S = np.zeros(N)
    for i in range(N):
        a, b2 = max(0, i - k), min(N - 1, i + k); dist = max(ds[a + 1:b2 + 1].sum(), 1.0); S[i] = (bed[a] - bed[b2]) / dist
    S = np.clip(S, 0.003, 0.3); R = Ac / np.maximum(width + 2 * depth, 1.0)
    v = np.clip((1 / n) * R ** (2 / 3) * np.sqrt(S), 1.0, 12.0)
    t_min = np.cumsum(ds / v) / 60.0
    return dict(mask=mask, path_used=used, t_min=t_min[:used], dist_m=np.cumsum(ds)[:used], depth=depth[:used], bed=bed[:used],
                A_flow=Ac, B_plan=Bmax, extent_km2=float(mask.sum() * px * px / 1e6))

def haversine_m(la1, lo1, la2, lo2):
    p = math.pi / 180
    a = math.sin((la2 - la1) * p / 2) ** 2 + math.cos(la1 * p) * math.cos(la2 * p) * math.sin((lo2 - lo1) * p / 2) ** 2
    return 12742000 * math.asin(math.sqrt(a))

def exposure(res, path, x0, y0, px, epsg, places, roads):
    """places: [(name, lat, lng)], roads: [[(lat,lng),...]].
    Strict: settlement within 90 m of the flood footprint. Also reports the NEAREST settlement (distance to the footprint,
    time the flood front needs to pass it) and how many settlements lie within 500 m (30 m DEM: villages sit on terraces)."""
    from pyproj import Transformer
    tr = Transformer.from_crs(4326, epsg, always_xy=True); mask = res["mask"]; H, W = mask.shape
    thin = ndi.binary_dilation(mask, iterations=1)
    dist_m = ndi.distance_transform_edt(~mask) * px if mask.any() else np.full(mask.shape, np.inf)
    P = np.array(path[:res["path_used"]], float); tree = cKDTree(P)
    hit, times, found = [], [], []
    for name, la, lo in places:
        X, Y = tr.transform(lo, la); c, r = (X - x0) / px, (y0 - Y) / px
        if not (0 <= r < H and 0 <= c < W): continue
        d = float(dist_m[int(r), int(c)]); t = float(res["t_min"][tree.query([r, c])[1]]); found.append((d, name, t))
        if d <= 90: hit.append(name); times.append(t)
    km = 0.0
    for way in roads:
        for (a, b) in zip(way, way[1:]):
            X, Y = tr.transform((a[1] + b[1]) / 2, (a[0] + b[0]) / 2); c, r = (X - x0) / px, (y0 - Y) / px
            if 0 <= r < H and 0 <= c < W and thin[int(r), int(c)]: km += haversine_m(a[0], a[1], b[0], b[1]) / 1000
    near = min(found) if found else None
    return dict(settlements=len(set(hit)), names=sorted(set(hit))[:12], roads_km=round(km, 1), t_first_min=min(times) if times else None,
                within500=len({n for d, n, t in found if d <= 500}),
                nearest=None if near is None else dict(name=near[1], dist_m=round(near[0]), t_min=round(near[2])))

OSM_ERR = []
_HEAD = {"User-Agent": "HIMNETRA-student-project/1.0 (college major project; glacial lake flood exposure)"}
_MIRRORS = ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter",
            "https://overpass.private.coffee/api/interpreter"]

def _post(q, tries=4):
    import requests
    for t in range(tries):
        for url in _MIRRORS:
            try:
                r = requests.post(url, data={"data": q}, headers=_HEAD, timeout=150)
                if r.status_code == 200 and "json" in r.headers.get("content-type", ""):
                    j = r.json()
                    if j.get("remark") and not j.get("elements"):          # Overpass says 200 but the query timed out / ran out of memory
                        OSM_ERR.append(f"{url.split('/')[2]} remark: {str(j['remark'])[:60]}"); continue
                    return j
                OSM_ERR.append(f"{url.split('/')[2]} HTTP {r.status_code}")
                if r.status_code in (429, 504): time.sleep(8 * (t + 1))
            except Exception as e: OSM_ERR.append(f"{url.split('/')[2]} {type(e).__name__}")
        time.sleep(5 * (t + 1))
    return None

def overpass(path_ll, radius=1000, cache_key=None, cache_file=None, chunk=12):
    """Settlements and roads within `radius` m of the flood path (list of (lat, lng)).
    The path is queried in small pieces (~5 km each): one big query makes Overpass time out and return EMPTY results."""
    import os
    cache = json.load(open(cache_file)) if (cache_file and os.path.exists(cache_file)) else {}
    if cache_key and cache_key in cache:
        c = cache[cache_key]; return [tuple(p) for p in c["places"]], [[tuple(q) for q in w] for w in c["roads"]]
    places, roads, seen = [], [], set()
    pieces = [path_ll[i:i + chunk + 1] for i in range(0, max(len(path_ll) - 1, 1), chunk)]
    for piece in pieces:
        pts = ",".join(f"{la:.5f},{lo:.5f}" for la, lo in piece)
        q = (f'[out:json][timeout:60];(node["place"~"village|hamlet|town|city|isolated_dwelling"](around:{radius},{pts});'
             f'way["highway"~"motorway|trunk|primary|secondary|tertiary|unclassified|residential|track"](around:{radius},{pts}););out geom;')
        j = _post(q)
        if j is None: return None, None
        for e in j.get("elements", []):
            if (e["type"], e["id"]) in seen: continue
            seen.add((e["type"], e["id"]))
            if e["type"] == "node": places.append((e.get("tags", {}).get("name", f"place-{e['id']}"), e["lat"], e["lon"]))
            elif e["type"] == "way" and "geometry" in e: roads.append([(g["lat"], g["lon"]) for g in e["geometry"]])
        time.sleep(1.0)
    if cache_key and cache_file:
        cache[cache_key] = dict(places=places, roads=roads); json.dump(cache, open(cache_file, "w"))
    return places, roads


In [ ]:
#@title 4) Self-test on a synthetic valley (must end with ALL OK)
from affine import Affine
px = 30.0; H = W = 600
rows, cols = np.mgrid[:H, :W].astype("float32")
dem_t = 4200 - 0.06 * rows * px + 0.25 * np.abs(cols - 300) * px
dem_t[(rows > 10) & (rows < 40) & (np.abs(cols - 300) < 6)] -= 15
filled_t, fdir_t = condition_dem(dem_t, Affine(px, 0, 300000., 0, -px, 3600000.), 32643)
path_t = trace_path(fdir_t, 25, 300, 1000)
res_t = inundate(filled_t, path_t, px, 20e6)
w_an = 8 * math.sqrt(A_COEF * (20e6) ** (2 / 3) / 4) / px
print("path cells:", len(path_t), "| flood width px (model / analytic):", res_t["mask"][200].sum(), "/", round(w_an, 1))
assert len(path_t) > 300 and abs(res_t["mask"][200].sum() - w_an) < 3
print("ALL OK")


In [ ]:
#@title 5) Per-lake runner, map helpers, plot (just run)
from affine import Affine
from pyproj import Transformer
import rasterio.features
from shapely.geometry import shape, mapping
from shapely.ops import transform as shp_transform, unary_union
import matplotlib; import matplotlib.pyplot as plt
from matplotlib.colors import LightSource, ListedColormap

def epsg_for(lng): return 32600 + int((lng + 180) // 6) + 1
def to_ll(epsg): return Transformer.from_crs(epsg, 4326, always_xy=True).transform
def with_retry(fn, tries=3, wait=6):
    for t in range(tries):
        try: return fn()
        except Exception:
            if t == tries - 1: raise
            time.sleep(wait * (t + 1))

def fetch_dem(lat, lng, half_px=None, scale=30):
    half_px = half_px or DEM_HALF_PX; epsg = epsg_for(lng)
    X, Y = Transformer.from_crs(4326, epsg, always_xy=True).transform(lng, lat)
    n = 2 * half_px; x0 = round(X / scale) * scale - half_px * scale; y0 = round(Y / scale) * scale + half_px * scale
    roi = ee.Geometry.Rectangle([x0, y0 - n * scale, x0 + n * scale, y0], proj=f"EPSG:{epsg}", geodesic=False)
    img = ee.ImageCollection("COPERNICUS/DEM/GLO30").filterBounds(roi).select("DEM").mosaic().rename("DEM").toFloat()
    req = dict(expression=img, fileFormat="NPY",
               grid=dict(dimensions=dict(width=n, height=n),
                         affineTransform=dict(scaleX=scale, shearX=0, translateX=x0, shearY=0, scaleY=-scale, translateY=y0),
                         crsCode=f"EPSG:{epsg}"))
    arr = ee.data.computePixels(req)
    if isinstance(arr, (bytes, bytearray)): arr = np.load(io.BytesIO(arr))
    return np.asarray(arr["DEM"], dtype="float32"), dict(epsg=int(epsg), x0=float(x0), y0=float(y0), px=float(scale))

def mask_to_feature(mask, g, props):
    tr = Affine(g["px"], 0, g["x0"], 0, -g["px"], g["y0"]); back = to_ll(g["epsg"])
    geoms = [shp_transform(back, shape(s)) for s, v in rasterio.features.shapes(mask.astype("uint8"), mask=mask, transform=tr) if v == 1]
    return dict(type="Feature", geometry=mapping(unary_union(geoms).simplify(5e-5)), properties=props)

def run_lake(lake, volumes, use_osm=True):
    """volumes: {scenario_name: released volume in m3}. Returns (results, runs, context)."""
    c = lake["coordinates"]; dem, g = with_retry(lambda: fetch_dem(c["lat"], c["lng"]))
    if (dem <= 0).mean() > 0.05: print("  ⚠ DEM has gaps (>5% zeros)")
    filled, fdir = condition_dem(dem, Affine(g["px"], 0, g["x0"], 0, -g["px"], g["y0"]), g["epsg"])
    mid = dem.shape[0] // 2
    path = trace_path(fdir, mid, mid, int(MAX_RUN_KM * 1000 / g["px"] * 1.4))
    skip_m = 1.2 * math.sqrt(max(lake.get("currentArea", 0.0), 0.01) * 1e6 / math.pi)     # lake radius: the lake itself is not 'flood'
    runs = {k: inundate(filled, path, g["px"], v, skip_m=skip_m) for k, v in volumes.items()}
    big = max(runs.values(), key=lambda r: r["mask"].sum()); places = roads = None
    if use_osm and big["mask"].any():
        back = to_ll(g["epsg"]); idx = list(range(0, big["path_used"], 15)) + [big["path_used"] - 1]
        path_ll = [back(g["x0"] + (path[i][1] + .5) * g["px"], g["y0"] - (path[i][0] + .5) * g["px"])[::-1] for i in idx]
        del OSM_ERR[:]
        places, roads = overpass(path_ll, 1000, lake.get("id"), f"{OUT}/osm_cache_v2.json")
        if places is not None: print(f"  OSM corridor: {len(places)} places, {len(roads)} road ways")
        if places is None: print("  ⚠ OpenStreetMap unreachable -> settlements/roads left empty (not zero). Last errors:", OSM_ERR[-3:])
    out = {}
    for k, r in runs.items():
        ex = exposure(r, path, g["x0"], g["y0"], g["px"], g["epsg"], places, roads) if places is not None else None
        out[k] = dict(floodExtent=round(r["extent_km2"], 2),
                      timeToImpact=int(round(ex["t_first_min"])) if ex and ex["t_first_min"] is not None else None,
                      affectedSettlements=ex["settlements"] if ex else None, affectedRoads=ex["roads_km"] if ex else None,
                      releaseMm3=round(volumes[k] / 1e6, 2), peakDischarge=round(peak_discharge(volumes[k])),
                      reachKm=round(float(r["dist_m"][-1]) / 1000, 1), settlementNames=ex["names"] if ex else [],
                      osmPlaces=len(places) if places is not None else None, osmRoadWays=len(roads) if roads is not None else None,
                      settlementsWithin500m=ex["within500"] if ex else None,
                      nearestSettlement=ex["nearest"]["name"] if ex and ex["nearest"] else None,
                      nearestSettlementDistM=ex["nearest"]["dist_m"] if ex and ex["nearest"] else None,
                      nearestSettlementMin=ex["nearest"]["t_min"] if ex and ex["nearest"] else None)
    return out, runs, dict(dem=dem, g=g, path=path, places=places)

def plot_run(title, runs, ctx):
    dem, g, path = ctx["dem"], ctx["g"], np.array(ctx["path"])
    r0, r1 = max(path[:, 0].min() - 80, 0), min(path[:, 0].max() + 80, dem.shape[0])
    c0, c1 = max(path[:, 1].min() - 80, 0), min(path[:, 1].max() + 80, dem.shape[1])
    hs = LightSource(azdeg=315, altdeg=40).hillshade(dem[r0:r1, c0:c1], vert_exag=2, dx=g["px"], dy=g["px"])
    fig, ax = plt.subplots(1, 2, figsize=(17, 7), gridspec_kw=dict(width_ratios=[1.4, 1]))
    ax[0].imshow(hs, cmap="gray", extent=(c0, c1, r1, r0))
    for (k, r), col in zip(list(runs.items())[::-1], ["#7f1d1d", "#dc2626", "#f97316", "#facc15"][-len(runs):][::-1]):
        m = r["mask"][r0:r1, c0:c1]; ax[0].imshow(np.ma.masked_where(~m, m), cmap=ListedColormap([col]), alpha=.75, extent=(c0, c1, r1, r0))
    ax[0].plot(path[:, 1], path[:, 0], "c-", lw=.8); ax[0].plot(*path[0][::-1], "w*", ms=14)
    if ctx["places"]:
        back = Transformer.from_crs(4326, g["epsg"], always_xy=True)
        for nm, la, lo in ctx["places"]:
            X, Y = back.transform(lo, la); c, r = (X - g["x0"]) / g["px"], (g["y0"] - Y) / g["px"]
            if c0 < c < c1 and r0 < r < r1: ax[0].plot(c, r, "b^", ms=4)
    ax[0].set_title(f"{title}: flood extent per scenario (white star = lake outlet, blue = OSM places)"); ax[0].axis("off")
    r = list(runs.values())[-1]; d = r["dist_m"] / 1000
    ax[1].plot(d, r["bed"], "k-", label="channel bed (m)"); ax[1].set_xlabel("distance downstream (km)"); ax[1].set_ylabel("elevation (m)")
    a2 = ax[1].twinx(); a2.plot(d, r["t_min"], "r--", label="travel time (min)"); a2.set_ylabel("travel time (min)")
    ax[1].set_title("Valley profile and front travel time"); plt.tight_layout(); plt.show()


In [ ]:
#@title 5b) Test OpenStreetMap access (must show places > 0)
del OSM_ERR[:]
t0 = time.time()
pl, ro = overpass([(32.570, 77.030), (32.572, 77.032)], radius=3000)      # Keylong area
print("places:", None if pl is None else len(pl), "| roads:", None if ro is None else len(ro), f"| {time.time()-t0:.0f}s", "| errors:", OSM_ERR[-4:])
assert pl is not None, "❌ Overpass unreachable from Colab right now - wait a few minutes and re-run this cell (public servers rate-limit shared IPs)"
print("✅ OSM reachable")


In [ ]:
#@title 6) ONE lake first (Gepang Gath): look at the picture before running everything
l = next(x for x in lakes if x["id"] == "GL-HP-001")
V = lake_volume_m3(l["currentArea"]); print(f"{l['name']}: area {l['currentArea']} km2 -> lake volume ~{V/1e6:.1f} M m3")
out, runs, ctx = run_lake(l, {k: V * f for k, f in FRACTION.items()})
print("channel path:", len(ctx["path"]), "cells =", round(len(ctx["path"]) * 30 / 1000, 1), "km (must be several km, otherwise the outlet is wrong)")
for k, v in out.items(): print(f"{k:9}", v)
plot_run(l["name"], runs, ctx)
print("👀 The cyan line must run down the valley away from the lake and the coloured flood must hug the valley floor.")


In [ ]:
#@title 7) Run all lakes x 4 scenarios (resumable; ~1 min per lake)
RES_F, POLY_F = f"{OUT}/flood_results.json", f"{OUT}/flood_polygons.geojson"
results = json.load(open(RES_F)) if os.path.exists(RES_F) else {}
feats = json.load(open(POLY_F))["features"] if os.path.exists(POLY_F) else []
todo = [l for l in sorted(lakes, key=lambda d: -d["riskScore"]) if (not ONLY or l["id"] in ONLY)]
for l in todo:
    if l["id"] in results and not FORCE and results[l["id"]]["MAJOR"].get("settlementsWithin500m") is not None: continue   # skip only lakes finished with the current logic; redo old / failed ones
    t0 = time.time()
    try:
        V = lake_volume_m3(l["currentArea"])
        out, runs, ctx = run_lake(l, {k: V * f for k, f in FRACTION.items()})
        if out["MAJOR"]["reachKm"] < 3:
            print(f"{l['id']} {l['name'][:24]:24} ⚠ flow path only {out['MAJOR']['reachKm']} km (outlet routing failed) -> skipped, UI will show a dash"); results.pop(l["id"], None); continue
        results[l["id"]] = out
        new = [mask_to_feature(r["mask"], ctx["g"], dict(lakeId=l["id"], scenario=k, extent_km2=out[k]["floodExtent"])) for k, r in runs.items() if r["mask"].any()]
        feats = [f for f in feats if f["properties"]["lakeId"] != l["id"]] + new
        json.dump(results, open(RES_F, "w"), indent=1); json.dump(dict(type="FeatureCollection", features=feats), open(POLY_F, "w"))
        m = out["MAJOR"]; print(f"{l['id']} {l['name'][:24]:24} reach {m['reachKm']:5} km | MAJOR extent {m['floodExtent']:6} km2 | settlements {m['affectedSettlements']} | roads {m['affectedRoads']} km | first impact {m['timeToImpact']} min | nearest {m['nearestSettlement']} {m['nearestSettlementDistM']} m / {m['nearestSettlementMin']} min | OSM {m['osmPlaces']}p/{m['osmRoadWays']}w ({time.time()-t0:.0f}s)")
    except Exception as e:
        print(f"{l['id']} FAILED: {str(e)[:150]}")
    time.sleep(2)
print("✅", len(results), "lakes in flood_results.json,", len(feats), "flood polygons")


In [ ]:
#@title 8) South Lhonak back-test: release the volume the lake actually lost on 3-4 Oct 2023
snap = json.load(open(f"{OUT}/snapped_lakes.json"))["BT-SK-001"]
ba = json.load(open(f"{OUT}/backtest_before_after.json"))
a_before = next(v["area_km2"] for k, v in ba.items() if k.startswith("before")); a_after = next(v["area_km2"] for k, v in ba.items() if k.startswith("after"))
dV = lake_volume_m3(a_before) - lake_volume_m3(a_after)
print(f"area before {a_before} km2, after {a_after} km2 -> volume lost ~{dV/1e6:.1f} M m3 (area-volume relation; compare with published estimates of the released water)")
bt = dict(coordinates=dict(lat=snap["lat"], lng=snap["lng"]), currentArea=a_before)
out, runs, ctx = run_lake(bt, {"OBSERVED": dV})
print(out["OBSERVED"]); plot_run("South Lhonak (Sikkim) back-test", runs, ctx)
json.dump(dict(areaBefore=a_before, areaAfter=a_after, volumeLostMm3=round(dV / 1e6, 1), result=out["OBSERVED"]), open(f"{OUT}/flood_backtest.json", "w"), indent=1)
print("Compare the reach and flood path with the published account of the 2023 event (Lhonak Chu -> Teesta, Chungthang). Report agreement AND disagreement honestly.")


In [ ]:
#@title 9) Summary + download
import shutil
print("Results:"); print(f"{'lake':10} {'name':24} {'MINOR':>7} {'MODERATE':>9} {'MAJOR':>7} {'EXTREME':>8}   (flood extent km2)")
for l in lakes:
    r = json.load(open(RES_F)).get(l["id"])
    if r: print(f"{l['id']:10} {l['name'][:24]:24} " + " ".join(f"{r[k]['floodExtent']:>8}" for k in ["MINOR", "MODERATE", "MAJOR", "EXTREME"]))
shutil.make_archive("/content/himnetra_flood", "zip", OUT)
try:
    from google.colab import files; files.download("/content/himnetra_flood.zip")
except Exception as e: print("download from Drive:", OUT)
print("✅ Copy flood_results.json and flood_polygons.geojson into backend/data/ (flood_backtest.json optional).")
